# Practical Exam: Customer Purchase Prediction

RetailTech Solutions is a fast-growing international e-commerce platform operating in over 20 countries across Europe, North America, and Asia. They specialize in fashion, electronics, and home goods, with a unique business model that combines traditional retail with a marketplace for independent sellers.

The company has seen rapid growth. A key part of their success has been their data-driven approach to personalization. However, as they plan their expansion into new markets, they need to improve their ability to predict customer behavior.

Their marketing team wants to predict which customers are most likely to make a purchase based on their browsing behavior.

As an AI Engineer, you will help build this prediction system. Your work will directly impact RetailTech's growth strategy and their goal of increasing revenue.


## Data Description

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

# Task 1

The marketing team has collected customer session data in `raw_customer_data.csv`, but it contains missing values and inconsistencies that need to be addressed.
Create a cleaned version of the dataframe:

- Start with the data in the file `raw_customer_data.csv`
- Your output should be a DataFrame named `clean_data`
- All column names and values should match the table below.
</br>

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

In [11]:
# Write your answer to Task 1 here 
import pandas as pd

# Memuat data mentah
clean_data = pd.read_csv('raw_customer_data.csv')

# Mengisi nilai yang hilang pada 'time_spent' dengan nilai median
median_time = clean_data['time_spent'].median()
clean_data['time_spent'] = clean_data['time_spent'].fillna(median_time).astype(float)

# Mengisi nilai yang hilang pada 'pages_viewed' dengan nilai rata-rata (mean)
mean_pages = clean_data['pages_viewed'].mean()
clean_data['pages_viewed'] = clean_data['pages_viewed'].fillna(mean_pages).astype(int)

# Mengisi nilai yang hilang pada 'basket_value' dengan 0
clean_data['basket_value'] = clean_data['basket_value'].fillna(0).astype(float)

# Mengisi nilai yang hilang pada 'device_type' dengan "Unknown"
clean_data['device_type'] = clean_data['device_type'].fillna("Unknown").astype(str)

# Mengisi nilai yang hilang pada 'customer_type' dengan "New"
clean_data['customer_type'] = clean_data['customer_type'].fillna("New").astype(str)

# Memastikan kolom 'customer_id' dan 'purchase' tidak berubah dan tipenya sesuai
clean_data['customer_id'] = clean_data['customer_id'].astype(int)
clean_data['purchase'] = clean_data['purchase'].astype(int)

# Melihat 5 baris pertama dan info tipe data untuk memastikan sudah sesuai
print(clean_data.head())
clean_data.info()


   customer_id  time_spent  pages_viewed  ...  device_type customer_type purchase
0            1   23.097867             7  ...       Mobile     Returning        0
1            2   57.092144             3  ...       Mobile     Returning        1
2            3   44.187643            14  ...       Mobile     Returning        0
3            4   36.320851            10  ...       Mobile           New        1
4            5   10.205100            16  ...       Mobile     Returning        1

[5 rows x 7 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    500 non-null    int64  
 1   time_spent     500 non-null    float64
 2   pages_viewed   500 non-null    int64  
 3   basket_value   500 non-null    float64
 4   device_type    500 non-null    object 
 5   customer_type  500 non-null    object 
 6   purchase       500 non-nu

# Task 2
The pre-cleaned dataset `model_data.csv` needs to be prepared for our neural network.
Create the model features:

- Start with the data in the file `model_data.csv`
- Scale numerical features (`time_spent`, `pages_viewed`, `basket_value`) to 0-1 range
- Apply one-hot encoding to the categorical features (`device_type`, `customer_type`)
    - The column names should have the following format: variable_name_category_name (e.g., `device_type_Desktop`)
- Your output should be a DataFrame named `model_feature_set`, with all column names from `model_data.csv` except for the columns where one-hot encoding was applied.


In [13]:
# Write your answer to Task 2 here
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Memuat dataset yang akan disiapkan
model_data = pd.read_csv('model_data.csv')

# Inisialisasi MinMaxScaler untuk mengubah rentang fitur menjadi 0 - 1
scaler = MinMaxScaler()
num_cols = ['time_spent', 'pages_viewed', 'basket_value']

# Menerapkan scaling pada kolom numerik
model_data[num_cols] = scaler.fit_transform(model_data[num_cols])

# Menerapkan One-Hot Encoding pada fitur kategorikal
# pd.get_dummies secara otomatis menghapus kolom asli dan membuat kolom baru 
# dengan format nama_variabel_nama_kategori (misal: device_type_Mobile)
cat_cols = ['device_type', 'customer_type']
model_feature_set = pd.get_dummies(model_data, columns=cat_cols, prefix=cat_cols, prefix_sep='_')

# Melihat 5 baris pertama data yang sudah di-scale dan di-encode
print(model_feature_set.head())


   customer_id  time_spent  ...  customer_type_New  customer_type_Returning
0          501    0.664167  ...                  1                        0
1          502    0.483681  ...                  0                        1
2          503    0.231359  ...                  0                        1
3          504    0.792944  ...                  1                        0
4          505    0.649210  ...                  1                        0

[5 rows x 11 columns]


# Task 3

Now that all preparatory work has been done, create and train a neural network that would allow the company to predict purchases.

- Using PyTorch, create a network with:
   - At least one hidden layer with 8 units
   - ReLU activation for hidden layer
   - Sigmoid activation for the output layer
- Using the prepared features in `input_model_features.csv`, train the model to predict purchases. 
- Use the validation dataset `validation_features.csv` to predict new values based on the trained model. 
- Your model should be named `purchase_model` and your output should be a DataFrame named `validation_predictions` with columns `customer_id` and `purchase`. The `purchase` column must be your predicted values.


In [14]:
# Write your answer to Task 3 here
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Memuat data latih dan data validasi
train_data = pd.read_csv('input_model_features.csv')
val_data = pd.read_csv('validation_features.csv')

# Memisahkan fitur (X) dan target (y)
# Hapus kolom target dan ID dari fitur pelatihan
X_train = train_data.drop(columns=['purchase', 'customer_id'], errors='ignore').values
y_train = train_data['purchase'].values

# Simpan customer_id dari data validasi untuk output akhir
val_customer_ids = val_data['customer_id']
X_val = val_data.drop(columns=['customer_id'], errors='ignore').values

# Konversi data numpy array menjadi PyTorch Tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).view(-1, 1)
X_val_tensor = torch.FloatTensor(X_val)

# Membuat arsitektur model Neural Network (purchase_model)
class PurchasePredictor(nn.Module):
    def __init__(self, input_dim):
        super(PurchasePredictor, self).__init__()
        # Hidden layer dengan 8 unit
        self.layer1 = nn.Linear(input_dim, 8) 
        # Aktivasi ReLU untuk hidden layer
        self.relu = nn.ReLU()                 
        self.output = nn.Linear(8, 1)
        # Aktivasi Sigmoid untuk output layer (klasifikasi biner)
        self.sigmoid = nn.Sigmoid()           

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.output(x)
        x = self.sigmoid(x)
        return x

input_size = X_train.shape[1]
purchase_model = PurchasePredictor(input_size)

# Konfigurasi fungsi Loss dan Optimizer
criterion = nn.BCELoss() # Binary Cross Entropy untuk target klasifikasi 0 dan 1
optimizer = optim.Adam(purchase_model.parameters(), lr=0.01)

# Melatih model (Training Loop)
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()                 # Reset gradien
    outputs = purchase_model(X_train_tensor) # Forward pass
    loss = criterion(outputs, y_train_tensor)# Hitung loss
    loss.backward()                       # Backward pass (hitung gradien)
    optimizer.step()                      # Update bobot

# Memprediksi data validasi
purchase_model.eval() # Set model ke mode evaluasi
with torch.no_grad():
    predictions = purchase_model(X_val_tensor)
    # Ubah probabilitas keluaran sigmoid (0.0-1.0) menjadi prediksi biner (0 atau 1)
    predicted_classes = (predictions >= 0.5).int().numpy().flatten()

# Menyimpan hasil prediksi dalam DataFrame yang diminta
validation_predictions = pd.DataFrame({
    'customer_id': val_customer_ids,
    'purchase': predicted_classes
})

# Melihat distribusi hasil prediksi
print(validation_predictions['purchase'].value_counts())
print(validation_predictions.head())

1    200
Name: purchase, dtype: int64
   customer_id  purchase
0         1801         1
1         1802         1
2         1803         1
3         1804         1
4         1805         1
